# Character training data generator
This notebook generates 42x42px PNG images of characters for training data.

In [5]:
import string
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from utils import preprocessing_remove_lines, preprocessing_split_by_characters

In [6]:
keys = list(string.ascii_lowercase) + [str(i) for i in range(10)]
char_dict = {k: [] for k in keys}
char_dir = 'chars_train_data/'
# Delete all files in the target folder
for filename in tqdm(os.listdir(char_dir), desc="Deleting old files"):
    file_path = os.path.join(char_dir, filename)
    if os.path.isfile(file_path) and filename.lower().endswith(".png"):
        os.remove(file_path)
        
os.makedirs(char_dir, exist_ok=True)

Deleting old files: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:00<?, ?it/s]


# Start character separation

In [7]:
wrong_count = 0
images_count = 0

# Loop through all png in train/
for (root,dirs,files) in os.walk('clean_train_data/',topdown=True):
    for file in tqdm(files, desc="Loading images"):
        if file.endswith('.png'):
            if file[1] == "_":
                os.remove(os.path.join(root, file))
                continue
            if file.split("-")[1].split(".png")[0] != "0":
                continue
                
            img_path = os.path.join(root, file)
            img = cv2.imread(img_path)
            processed_img = preprocessing_remove_lines(img) # Remove lines
            num_chars, char_images = preprocessing_split_by_characters(processed_img) # Split chars
            ground_truth = img_path.split("-")[0].split("/")[-1] # Get Captcha label

            # If wrong seperation count, do not add to chars_train_data
            images_count += 1
            if len(char_images) != len(ground_truth):
                wrong_count += 1
                continue

            # Correct seperated
            for i in range(len(ground_truth)):
                k = ground_truth[i]
                char_dict[k].append(char_images[i])

print(f"Total captcha count: {images_count}")
print(f"Wrong character sep count: {wrong_count}")
            

Loading images: 100%|██████████████████████████████████████████████████████████████| 8001/8001 [04:53<00:00, 27.26it/s]

Total captcha count: 7812
Wrong character sep count: 433


### Process character image after separation

In [8]:
def process_character_image(img):
    # --- Convert to grayscale ---
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # --- Invert so character = white (255), background = black (0) ---
    gray = 255 - gray
    
    # --- Threshold to make binary ---
    # _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # --- Find bounding box of character ---
    coords = cv2.findNonZero(gray)
    x, y, w, h = cv2.boundingRect(coords)
    
    # --- Crop + 3px padding ---
    pad = 3
    x1 = max(x - pad, 0)
    y1 = max(y - pad, 0)
    x2 = min(x + w + pad, gray.shape[1])
    y2 = min(y + h + pad, gray.shape[0])
    cropped = gray[y1:y2, x1:x2]

    # --- Normalize brightness: stretch intensity range to full 0–255 ---
    min_val, max_val = np.min(cropped), np.max(cropped)
    if max_val > min_val:  # avoid divide-by-zero if image is uniform
        cropped = (cropped - min_val) * (255.0 / (max_val - min_val))
        cropped = np.clip(cropped, 0, 255).astype(np.uint8)
    
    # Below: Pad or crop all letters to the same square (while avoiding stretching proportions)
    # --- Create black background ---
    target_size = 42
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    
    # --- If cropped image is larger than 32x32, center crop it ---
    h, w = cropped.shape
    if h > target_size or w > target_size:
        # Center crop
        start_y = max((h - target_size) // 2, 0)
        start_x = max((w - target_size) // 2, 0)
        end_y = start_y + target_size
        end_x = start_x + target_size
        cropped = cropped[start_y:end_y, start_x:end_x]
        h, w = cropped.shape
    
    # --- Compute top-left position to center the image ---
    y_offset = (target_size - h) // 2
    x_offset = (target_size - w) // 2
    
    # --- Paste cropped image onto canvas ---
    canvas[y_offset:y_offset + h, x_offset:x_offset + w] = cropped

    return canvas

### Save to PNG training data

In [9]:
# Save each image to PNG
for k, img_list in char_dict.items():
    for idx, img in tqdm(enumerate(img_list), desc=k):
        filename = f"{k}_{idx:05d}.png"  # e.g., a_0000.png
        path = os.path.join(char_dir, filename)
        processed_img = process_character_image(img)
        cv2.imwrite(path, processed_img)



a: 1186it [00:03, 377.52it/s]
b: 1255it [00:03, 381.66it/s]
c: 1217it [00:03, 380.87it/s]
d: 1265it [00:03, 378.84it/s]
e: 1258it [00:03, 377.10it/s]
f: 1229it [00:03, 372.35it/s]
g: 1253it [00:03, 362.28it/s]
h: 1238it [00:03, 357.49it/s]
i: 1216it [00:03, 357.82it/s]
j: 1173it [00:03, 336.55it/s]
k: 1226it [00:03, 354.55it/s]
l: 1212it [00:03, 334.84it/s]
m: 1235it [00:03, 365.11it/s]
n: 1279it [00:03, 381.81it/s]
o: 1211it [00:03, 369.77it/s]
p: 1252it [00:03, 396.98it/s]
q: 1273it [00:03, 394.41it/s]
r: 1221it [00:03, 396.56it/s]
s: 1201it [00:03, 389.66it/s]
t: 1221it [00:03, 403.55it/s]
u: 1185it [00:02, 399.15it/s]
v: 1235it [00:03, 371.57it/s]
w: 1200it [00:03, 373.88it/s]
x: 1265it [00:03, 368.27it/s]
y: 1187it [00:03, 392.79it/s]
z: 1219it [00:03, 394.88it/s]
0: 1232it [00:03, 385.88it/s]
1: 1239it [00:03, 395.35it/s]
2: 1169it [00:02, 393.13it/s]
3: 1243it [00:03, 388.09it/s]
4: 1225it [00:03, 395.88it/s]
5: 1181it [00:03, 390.46it/s]
6: 1220it [00:03, 389.58it/s]
7: 1164it 